# Part 2 of 4: Clean the data and build the new columns

**Input:** `wedmegood_hyderabad_venues_raw.csv` (from Part 1)  
**Output:** `wedmegood_clean.csv` (used by Parts 3 and 4)

**Main price:** the **rental cost per function** (the flat price for the venue; food, decor and services are extra).  
**Secondary price:** the **price per plate** (veg and non-veg).

Steps: load, remove repeats, turn text into numbers, build the rental cost, clean prices, split the location, group venue types, add helper columns, save.

In [8]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

RAW_FILE = "wedmegood_hyderabad_venues_raw.csv"
CLEAN_FILE = "wedmegood_clean.csv"

df = pd.read_csv(RAW_FILE)

# If a column is missing for any reason, add it empty so nothing breaks later
expected = ["venue_name", "detail_url", "page", "is_paid", "rating", "reviews", "location_text", "space_type",
            "veg_price", "nonveg_price", "venue_price_card", "capacity_min", "capacity_max", "rooms_card",
            "more_count", "rental_cost_detail", "rooms_detail"]
for col in expected:
    if col not in df.columns:
        print("Column missing in the raw file, added empty:", col)
        df[col] = np.nan

df["is_paid"] = df["is_paid"].fillna(False).astype(bool)
print("Rows and columns:", df.shape)
df.head(10)

Rows and columns: (500, 18)


,venue_name,detail_url,page,is_paid,rating,reviews,location_text,space_type,veg_price,nonveg_price,venue_price_card,capacity_min,capacity_max,rooms_card,more_count,rental_cost_detail,rooms_detail,detail_status
0,Advaya Convention,https://www.wedmegood.com/wedding-venues/Advay...,1,False,5.0,( 7 reviews ),"Medchal , Hyderabad Large Lawn with Banquet",NaN,NaN,NaN,NaN,500.0,10000.0,8.0,3,NaN,8.0,no_sentence
1,RajKamal Palace,https://www.wedmegood.com/wedding-venues/RajKa...,1,False,5.0,( 49 reviews ),"Kompally , Hyderabad European-Style Lawn",NaN,NaN,NaN,NaN,250.0,500.0,2.0,7,NaN,2.0,no_sentence
2,Arden Lakefront,https://www.wedmegood.com/wedding-venues/Arden...,1,False,5.0,( 3 reviews ),Hyderabad Lake View/River View,NaN,NaN,NaN,"1,25,000",100.0,500.0,4.0,4,"125,000",4.0,ok
3,Anamkara Resort by Mrugavani,https://www.wedmegood.com/wedding-venues/Anamk...,1,False,5.0,( 1 review ),Medak Mountain View Venue,NaN,"1,300","1,500",NaN,20.0,1500.0,40.0,4,"100,000",40.0,ok
4,Dream Valley Resorts,https://www.wedmegood.com/wedding-venues/Dream...,1,False,4.2,( 11 reviews ),"Moinabad , Hyderabad Lush Wedding Resort",NaN,"2,000","2,250",NaN,100.0,5000.0,90.0,3,NaN,90.0,no_sentence
5,Telangana Araku Resort,https://www.wedmegood.com/wedding-venues/Telan...,1,False,5.0,( 11 reviews ),Bodakonda Farmhouse With Banquet,NaN,NaN,NaN,"1,80,000",200.0,1000.0,15.0,4,"180,000",15.0,ok
6,The Ivory Courtyard,https://www.wedmegood.com/wedding-venues/The-I...,1,False,5.0,( 13 reviews ),"Moinabad , Hyderabad Lawn & Banquet Hall",NaN,750,950,NaN,350.0,800.0,7.0,5,"100,000",7.0,ok
7,Lohia Convention & Garden,https://www.wedmegood.com/wedding-venues/Lohia...,1,False,5.0,( 2 reviews ),"Miyapur , Hyderabad Convention Hall & Lawn",NaN,"1,650","2,250",NaN,200.0,1200.0,4.0,3,NaN,4.0,no_sentence
8,The Portico Boutique Resort,https://www.wedmegood.com/wedding-venues/The-P...,1,False,5.0,( 5 reviews ),"Shamshabad , Hyderabad Resort",NaN,NaN,NaN,"3,00,000",100.0,1200.0,10.0,4,"300,000",10.0,ok
9,Arcadia - Event Venue in Hyderabad,https://www.wedmegood.com/wedding-venues/Arcad...,1,False,5.0,( 4 reviews ),Hyderabad Wedding Farmhouse,NaN,NaN,NaN,"3,50,000",400.0,600.0,8.0,4,"350,000",8.0,ok


In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   venue_name          500 non-null    str    
 1   detail_url          500 non-null    str    
 2   page                500 non-null    int64  
 3   is_paid             500 non-null    bool   
 4   rating              321 non-null    float64
 5   reviews             500 non-null    str    
 6   location_text       500 non-null    str    
 7   space_type          272 non-null    str    
 8   veg_price           334 non-null    str    
 9   nonveg_price        314 non-null    str    
 10  venue_price_card    89 non-null     str    
 11  capacity_min        499 non-null    float64
 12  capacity_max        499 non-null    float64
 13  rooms_card          229 non-null    float64
 14  more_count          500 non-null    int64  
 15  rental_cost_detail  138 non-null    str    
 16  rooms_detail       

## 1. Remove repeated venues
Promoted (paid) venues can appear more than once. A repeat is the same venue page, or the same name in the same location.

In [11]:
before = len(df)
repeat_page = df["detail_url"].notna() & df["detail_url"].duplicated()
df = df[~repeat_page] #Keeps rows that are not marked as repeated URLs.
df = df.drop_duplicates(subset=["venue_name", "location_text"]).reset_index(drop=True)
print(f"Rows before: {before}, after removing repeats: {len(df)}")

Rows before: 500, after removing repeats: 500


## 2. Turn text into numbers
Right after scraping, everything is text: prices look like `1,80,000`, reviews look like `( 2 reviews )`. This pulls the first number out of each text.

In [13]:
def to_number(series):
    """Keep the first number found in the text (commas removed). Anything else becomes missing."""
    cleaned = series.astype(str).str.replace(",", "", regex=False).str.extract(r"(\d+\.?\d*)")[0]
    return pd.to_numeric(cleaned, errors="coerce")


number_cols = ["rating", "reviews", "veg_price", "nonveg_price", "venue_price_card", "rental_cost_detail",
               "capacity_min", "capacity_max", "rooms_card", "rooms_detail", "more_count"]
for col in number_cols:
    df[col] = to_number(df[col])

# A rating must be between 0 and 5
df.loc[(df["rating"] < 0) | (df["rating"] > 5), "rating"] = np.nan

# No review text means no reviews. A missing rating stays missing (it is not a zero rating).
df["reviews"] = df["reviews"].fillna(0).astype(int)

df[number_cols].describe().round(1)

,rating,reviews,veg_price,nonveg_price,venue_price_card,rental_cost_detail,capacity_min,capacity_max,rooms_card,rooms_detail,more_count
count,321.0,500.0,334.0,314.0,89.0,138.0,499.0,499.0,229.0,221.0,500.0
mean,4.7,2.2,2745.0,820.9,273146.1,240253.6,385.3,989.9,38.9,39.8,2.8
std,0.5,5.2,28093.0,583.8,525291.8,448960.8,549.8,1269.4,70.2,71.3,0.8
min,0.5,0.0,80.0,170.0,12000.0,3500.0,2.0,50.0,1.0,1.0,2.0
25%,4.8,0.0,350.0,450.0,45000.0,45000.0,120.0,300.0,4.0,4.0,2.0
50%,5.0,1.0,500.0,612.5,100000.0,100000.0,250.0,600.0,12.0,12.0,3.0
75%,5.0,2.0,750.0,900.0,270000.0,237500.0,430.0,1125.0,35.0,38.0,3.0
max,5.0,69.0,500000.0,3800.0,3500000.0,3500000.0,7000.0,10000.0,462.0,462.0,8.0


## 3. Rental cost (the main price)
The card is used first. The venue page fills the gaps. `rental_source` records where each value came from.

In [14]:
both = df[df["venue_price_card"].notna() & df["rental_cost_detail"].notna()]
if len(both) > 0:
    same = (both["venue_price_card"] == both["rental_cost_detail"]).mean()
    print(f"Venues with both sources: {len(both)}. Share where the two values match: {same:.0%}")

df["rental_cost"] = df["venue_price_card"].fillna(df["rental_cost_detail"])
df.loc[df["rental_cost"] <= 0, "rental_cost"] = np.nan
#ecord the source of the price
df["rental_source"] = np.select(
    [df["venue_price_card"].notna(), df["rental_cost_detail"].notna()],
    ["Listing card", "Venue page"],
    default="Not available",
)
print(df["rental_source"].value_counts().to_string())
print(f"Rental cost available for {df['rental_cost'].notna().mean():.0%} of venues")
print("Rental costs below 5,000 (kept, but worth a look):", int((df["rental_cost"] < 5000).sum()))

## Stop the program if fewer than 10 venues have a rental cost
# This prevents continuing the analysis with too little price data
if df["rental_cost"].notna().sum() < 10:
    raise SystemExit("STOP: fewer than 10 venues have a rental cost, so there is nothing to analyse. "
                     "Go back to Part 1 (Steps 3 and 4) and check that the data was collected properly.")

Venues with both sources: 86. Share where the two values match: 100%
rental_source
Not available    359
Listing card      89
Venue page        52
Rental cost available for 28% of venues
Rental costs below 5,000 (kept, but worth a look): 1


## 4. Plate prices: remove impossible values
A price per plate is in the hundreds or low thousands. Anything above ₹10,000 is a flat price typed in the plate field, so it is set to missing. The rental cost column already holds the right flat price when the venue page gave it.

In [15]:
for col in ["veg_price", "nonveg_price"]:
    too_high = df[col] > 10000
    print(f"{col}: {int(too_high.sum())} impossible values set to missing")
    df.loc[too_high, col] = np.nan
    df.loc[df[col] <= 0, col] = np.nan

has_plate = df["veg_price"].notna() | df["nonveg_price"].notna()
has_function = df["rental_cost"].notna()
df["has_rental_cost"] = has_function
df["pricing_type"] = np.select(
    [has_plate & has_function, has_function, has_plate],
    ["Both", "Per function only", "Per plate only"],
    default="Not listed",
)
print(df["pricing_type"].value_counts().to_string())

veg_price: 4 impossible values set to missing
nonveg_price: 0 impossible values set to missing
pricing_type
Per plate only       283
Per function only     93
Not listed            76
Both                  48


## 5. Capacity, rooms and extra features

In [16]:
df.loc[df["capacity_min"] <= 0, "capacity_min"] = np.nan
df.loc[df["capacity_max"] <= 0, "capacity_max"] = np.nan

# If minimum is bigger than maximum, the two were swapped
swap = df["capacity_min"] > df["capacity_max"]
print("Capacity ranges that were swapped:", int(swap.sum()))
df.loc[swap, ["capacity_min", "capacity_max"]] = df.loc[swap, ["capacity_max", "capacity_min"]].values

df["rooms"] = df["rooms_card"].fillna(df["rooms_detail"])
df["has_rooms"] = df["rooms"].notna()                      # a missing value means the venue does not show rooms
df["extra_features"] = df["more_count"].fillna(0)          # the N in "+N more"; no link means 0

print("Missing capacity:", int(df["capacity_max"].isna().sum()))
print("Venues showing rooms:", int(df["has_rooms"].sum()), "of", len(df))

Capacity ranges that were swapped: 0
Missing capacity: 1
Venues showing rooms: 229 of 500


## 6. Split the location text into area and venue type
`location_text` looks like `Miyapur , Hyderabad Banquet Hall`. Some venues have no area, only `Hyderabad Banquet Hall`.

In [17]:
parts = df["location_text"].astype(str).str.extract(r"^(?:(.+?)\s*,\s*)?Hyderabad\s+(.+)$")
df["area"] = parts[0].str.strip().str.title()
df["venue_type_raw"] = parts[1].str.strip()
# If the text did not follow the pattern, keep the whole text as the type description
df["venue_type_raw"] = df["venue_type_raw"].fillna(df["location_text"])

print("Venues with no area listed:", int(df["area"].isna().sum()), "of", len(df))
df[["location_text", "area", "venue_type_raw"]].head(6)

Venues with no area listed: 168 of 500


,location_text,area,venue_type_raw
0,"Medchal , Hyderabad Large Lawn with Banquet",Medchal,Large Lawn with Banquet
1,"Kompally , Hyderabad European-Style Lawn",Kompally,European-Style Lawn
2,Hyderabad Lake View/River View,NaN,Lake View/River View
3,Medak Mountain View Venue,NaN,Medak Mountain View Venue
4,"Moinabad , Hyderabad Lush Wedding Resort",Moinabad,Lush Wedding Resort
5,Bodakonda Farmhouse With Banquet,NaN,Bodakonda Farmhouse With Banquet


In [18]:
type_text = df["venue_type_raw"].astype(str).str.lower()
has = lambda word: type_text.str.contains(word, na=False)

df["venue_group"] = np.select(
    [has("farmhouse"), has("resort"), has("hotel"), has("convention"), has("lawn") & has("banquet"), has("lawn"), has("banquet")],
    ["Farmhouse", "Resort", "Hotel", "Convention Hall", "Banquet & Lawn", "Lawn", "Banquet Hall"],
    default="Other",
)
print(df["venue_group"].value_counts().to_string())
print()
print("What fell into 'Other' (check nothing important is hidden):")
print(df.loc[df["venue_group"] == "Other", "venue_type_raw"].value_counts().head(8).to_string())

venue_group
Banquet Hall       159
Convention Hall     93
Banquet & Lawn      70
Hotel               59
Lawn                50
Farmhouse           27
Other               25
Resort              17

What fell into 'Other' (check nothing important is hidden):
venue_type_raw
Party Restaurant / Lounge    10
Kalyana Mandapam              7
Small Function Hall           2
Hyderabad                     2
Lake View/River View          1
Medak Mountain View Venue     1
Regal Fort Wedding Venue      1
Pure Veg Venue                1


## 7. Helper columns (only the ones the analysis needs)

| New column | Why |
|---|---|
| `rental_tier` | Low, Mid, High: each holds one third of the priced venues |
| `capacity_group` | Compare small and large venues |
| `rating_group` | Ratings bunch near 5.0, so groups are easier to read |
| `review_group`, `enough_reviews` | A 5.0 from 1 review is not the same as a 5.0 from 40 reviews |
| `cost_per_guest` | Rental cost divided by maximum capacity |
| `log_rental_cost`, `log_capacity` | Costs and capacities are very skewed, so tests use the log scale |

In [19]:
df["rental_tier"] = pd.qcut(df["rental_cost"].rank(method="first"), q=3, labels=["Low", "Mid", "High"])
df["capacity_group"] = pd.cut(df["capacity_max"], bins=[0, 500, 1000, 3000, np.inf],
                              labels=["Up to 500", "501-1000", "1001-3000", "Above 3000"])
df["rating_group"] = pd.cut(df["rating"], bins=[0, 4.4, 4.9, 5.0], labels=["Below 4.5", "4.5 to 4.9", "5.0"])
df["review_group"] = pd.cut(df["reviews"], bins=[-1, 0, 1, 4, 9, np.inf], labels=["0", "1", "2 to 4", "5 to 9", "10+"])
df["enough_reviews"] = df["reviews"] >= 5
df["cost_per_guest"] = df["rental_cost"] / df["capacity_max"]
df["log_rental_cost"] = np.log10(df["rental_cost"])
df["log_capacity"] = np.log10(df["capacity_max"])

for col in ["rental_tier", "capacity_group", "rating_group", "review_group"]:
    print(df[col].value_counts(dropna=False).sort_index().to_string(), "\n")

print("What each rental tier means in rupees:")
print(df.groupby("rental_tier", observed=True)["rental_cost"].agg(["min", "max", "count"]).to_string())

rental_tier
NaN     359
Low      47
Mid      47
High     47 

capacity_group
Up to 500     222
501-1000      150
1001-3000     105
Above 3000     22
NaN             1 

rating_group
Below 4.5      55
4.5 to 4.9     51
5.0           215
NaN           179 

review_group
0         179
1         181
2 to 4     80
5 to 9     33
10+        27 

What each rental tier means in rupees:
                  min        max  count
rental_tier                            
Low            3500.0    50000.0     47
Mid           55000.0   175000.0     47
High         180000.0  3500000.0     47


## 8. Final check and save

In [20]:
final_cols = ["venue_name", "detail_url", "page", "is_paid", "rating", "reviews", "area", "venue_type_raw", "venue_group",
              "space_type", "veg_price", "nonveg_price", "rental_cost", "rental_source", "pricing_type", "has_rental_cost",
              "capacity_min", "capacity_max", "rooms", "has_rooms", "extra_features", "rental_tier", "capacity_group",
              "rating_group", "review_group", "enough_reviews", "cost_per_guest", "log_rental_cost", "log_capacity"]
clean = df[final_cols].copy()

print("Rows and columns:", clean.shape)
print("\nShare of venues with a value (higher is better):")
print((clean[["rental_cost", "veg_price", "capacity_max", "rooms", "rating", "area", "space_type"]].notna().mean() * 100).round(0).astype(int).astype(str).add("%").to_string())

clean.to_csv(CLEAN_FILE, index=False)
print(f"\nSaved {CLEAN_FILE}. Close this notebook and open Part 3.")

Rows and columns: (500, 29)

Share of venues with a value (higher is better):
rental_cost      28%
veg_price        66%
capacity_max    100%
rooms            46%
rating           64%
area             66%
space_type       54%

Saved wedmegood_clean.csv. Close this notebook and open Part 3.


In [35]:
clean.drop(columns=[
    'rental_tier',
    'rental_source',
    'pricing_type',
    'has_rental_cost',
    'has_rooms',
    'is_paid',
    'extra_features',
    'rating_group',
    'enough_reviews',
    'cost_per_guest',
    'log_rental_cost',
    'log_capacity',
    'space_type',
    'review_group'
])

,venue_name,detail_url,page,rating,reviews,area,venue_type_raw,venue_group,veg_price,nonveg_price,rental_cost,capacity_min,capacity_max,rooms,capacity_group
0,Advaya Convention,https://www.wedmegood.com/wedding-venues/Advay...,1,5.0,7,Medchal,Large Lawn with Banquet,Banquet & Lawn,NaN,NaN,NaN,500.0,10000.0,8.0,Above 3000
1,RajKamal Palace,https://www.wedmegood.com/wedding-venues/RajKa...,1,5.0,49,Kompally,European-Style Lawn,Lawn,NaN,NaN,NaN,250.0,500.0,2.0,Up to 500
2,Arden Lakefront,https://www.wedmegood.com/wedding-venues/Arden...,1,5.0,3,NaN,Lake View/River View,Other,NaN,NaN,125000.0,100.0,500.0,4.0,Up to 500
3,Anamkara Resort by Mrugavani,https://www.wedmegood.com/wedding-venues/Anamk...,1,5.0,1,NaN,Medak Mountain View Venue,Other,1300.0,1500.0,100000.0,20.0,1500.0,40.0,1001-3000
4,Dream Valley Resorts,https://www.wedmegood.com/wedding-venues/Dream...,1,4.2,11,Moinabad,Lush Wedding Resort,Resort,2000.0,2250.0,NaN,100.0,5000.0,90.0,Above 3000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,Kothaapu Narsimha Reddy Banquet & Conference Hall,https://www.wedmegood.com/wedding-venues/Kotha...,26,NaN,0,Champapet,Banquet Hall,Banquet Hall,400.0,500.0,NaN,300.0,400.0,NaN,Up to 500
496,Baddam Surender Reddy,https://www.wedmegood.com/wedding-venues/Badda...,26,NaN,0,NaN,Convention Hall & Lawn,Convention Hall,NaN,NaN,100000.0,1500.0,2250.0,NaN,1001-3000
497,Hotel Suprabhat,https://www.wedmegood.com/wedding-venues/Hotel...,26,5.0,3,Tarnaka,Banquet Hall,Banquet Hall,350.0,NaN,NaN,250.0,350.0,NaN,Up to 500
498,Habeeb Function Hall,https://www.wedmegood.com/wedding-venues/Habee...,26,NaN,0,Falaknuma,Banquet Hall,Banquet Hall,NaN,NaN,NaN,250.0,400.0,NaN,Up to 500
